## Common

### Installations


In [86]:
%pip install -q "transformers>=4.46,<5" "datasets>=3,<4" accelerate jiwer soundfile scipy
%pip uninstall -y transformers
%pip install --no-cache-dir --force-reinstall "transformers==4.46.3"


SyntaxError: invalid syntax (2651505124.py, line 2)

### Imports and Configuration - Global

In [32]:

from pathlib import Path
import unicodedata
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, Wav2Vec2Config
import sys
import platform
import torch
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/Code-SwitchingASR/data") #Root directory in gdrive, with the data extracted into data folder
PROCESSED = ROOT / "processed"
RUNS = ROOT / "experiments"
CHECKPOINTS = RUNS / "checkpoints"
RESULTS = RUNS / "results"

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

print("\nDataset root exists:", ROOT.exists())
print("Processed directory exists:", PROCESSED.exists())

if not ROOT.exists():
    raise FileNotFoundError(f"Dataset root not found: {ROOT}")

for name in [
    "unified_manifest_hiacc_openslr.csv",
    "normalized_transcript_manifest.csv",
    "preprocessed_manifest.csv",
    "hiacc_rebuild_audit.csv",
    "hiacc_unmatched_rows.csv",
    "rejected_rows.csv",
    "local_text_composition_audit.csv",
]:
    path = PROCESSED / name
    print(f"{name}: {path.exists()}")

print("\nTransformers:", __import__("transformers").__version__)
print("Accelerate:", __import__("accelerate").__version__)
print("JIWER:", __import__("jiwer").__version__ if hasattr(__import__("jiwer"), "__version__") else "installed")

Python: 3.13.16
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56

Dataset root exists: True
Processed directory exists: True
unified_manifest_hiacc_openslr.csv: True
normalized_transcript_manifest.csv: True
preprocessed_manifest.csv: True
hiacc_rebuild_audit.csv: True
hiacc_unmatched_rows.csv: True
rejected_rows.csv: True
local_text_composition_audit.csv: True

Transformers: 4.46.3
Accelerate: 1.15.0
JIWER: installed

COMMON CELL 1 COMPLETE


### Data Audit

In [20]:

from pathlib import Path
import pandas as pd
import numpy as np

manifest_path = PROCESSED / "unified_manifest_hiacc_openslr.csv"
df = pd.read_csv(manifest_path, low_memory=False)

print("MANIFEST SHAPE:", df.shape)
print("\nCOLUMNS:")
print(df.columns.tolist())

print("\nSAMPLE ROWS:")
display(df.head(3))

def find_col(candidates):
    lookup = {c.lower().strip(): c for c in df.columns}
    for name in candidates:
        if name in lookup:
            return lookup[name]
    for c in df.columns:
        normalized = c.lower().strip()
        if any(name in normalized for name in candidates):
            return c
    return None

id_col = find_col(["uid", "utterance_id", "utt_id", "id"])
audio_col = find_col(["audio_path", "wav_path", "file_path", "audio", "path"])
text_col = find_col(["transcript", "sentence", "text", "normalized_text"])
split_col = find_col(["split", "subset", "partition"])
source_col = find_col(["source", "corpus", "dataset"])
duration_col = find_col(["duration_sec", "duration", "audio_duration"])
recording_col = find_col(["recording_id", "record_id", "recording"])

print("\nDETECTED COLUMNS:")
for name, col in [
    ("ID", id_col), ("Audio", audio_col), ("Transcript", text_col),
    ("Split", split_col), ("Source", source_col),
    ("Duration", duration_col), ("Recording ID", recording_col)
]:
    print(f"{name}: {col}")

if source_col:
    print("\nROWS BY SOURCE:")
    print(df[source_col].fillna("<missing>").value_counts().to_string())

if split_col:
    print("\nROWS BY SPLIT:")
    print(df[split_col].fillna("<missing>").value_counts().to_string())

if id_col:
    ids = df[id_col].dropna().astype(str)
    print("\nDUPLICATE NON-EMPTY IDS:", int(ids.duplicated().sum()))
    print("MISSING IDS:", int(df[id_col].isna().sum()))

if text_col:
    texts = df[text_col].fillna("").astype(str).str.strip()
    print("\nEMPTY TRANSCRIPTS:", int(texts.eq("").sum()))
    print("WHITESPACE-ONLY / EMPTY TRANSCRIPTS:", int(texts.str.fullmatch(r"\s*").sum()))

if duration_col:
    durations = pd.to_numeric(df[duration_col], errors="coerce")
    print("\nDURATION AUDIT:")
    print("Unparseable/missing:", int(durations.isna().sum()))
    print("Zero or negative:", int((durations <= 0).sum()))
    print("Total hours:", round(durations.clip(lower=0).sum() / 3600, 3))
    print("Min / median / max seconds:",
          durations.min(), durations.median(), durations.max())

if audio_col:
    audio_values = df[audio_col].dropna().astype(str)
    print("\nAUDIO PATH EXAMPLES:")
    print(audio_values.head(5).to_list())

    def resolve_audio(value):
        p = Path(value)
        if p.is_absolute() and p.exists():
            return p
        candidates = [ROOT / value, ROOT.parent / value]
        for candidate in candidates:
            if candidate.exists():
                return candidate
        return None

    sample_values = audio_values.head(1000)
    resolved_sample = [resolve_audio(v) for v in sample_values]
    found = sum(p is not None for p in resolved_sample)
    print(f"Audio paths found in first {len(sample_values)} rows: {found}/{len(sample_values)}")

    if len(audio_values) <= 100000:
        resolved_all = [resolve_audio(v) for v in audio_values]
        missing = [v for v, p in zip(audio_values, resolved_all) if p is None]
        print("Missing audio paths across manifest:", len(missing))
        if missing:
            print("First 10 missing paths:", missing[:10])

if split_col and recording_col:
    leakage = (
        df.dropna(subset=[recording_col, split_col])
          .groupby(recording_col)[split_col]
          .nunique()
    )
    leaked_recordings = leakage[leakage > 1]
    print("\nRECORDINGS APPEARING IN MULTIPLE SPLITS:", len(leaked_recordings))
    if len(leaked_recordings):
        print("Examples:", leaked_recordings.index[:10].to_list())

MANIFEST SHAPE: (61077, 16)

COLUMNS:
['uid', 'dataset', 'source', 'cohort', 'split', 'source_split', 'speaker_id', 'recording_id', 'audio_path', 'start_s', 'end_s', 'text_raw', 'text', 'duration_s', 'sample_rate', 'channels']

SAMPLE ROWS:


,uid,dataset,source,cohort,split,source_split,speaker_id,recording_id,audio_path,start_s,end_s,text_raw,text,duration_s,sample_rate,channels
0,hiacc_adult_train_split_AD09002,hiacc,hiacc_adult,adult,train,train_split,NaN,AD09002,/content/drive/MyDrive/AI_PROJECTS/Code-Switch...,0.0,10.361187,So my favourite festival is Diwali. The reason...,So my favourite festival is Diwali. The reason...,10.361187,16000,1
1,hiacc_adult_train_split_AD09003,hiacc,hiacc_adult,adult,train,train_split,NaN,AD09003,/content/drive/MyDrive/AI_PROJECTS/Code-Switch...,0.0,14.138375,And हम लोगों को मतलब सब लोग बनाते हैं इसको और ...,And हम लोगों को मतलब सब लोग बनाते हैं इसको और ...,14.138375,16000,1
2,hiacc_adult_train_split_AD09005,hiacc,hiacc_adult,adult,train,train_split,NaN,AD09005,/content/drive/MyDrive/AI_PROJECTS/Code-Switch...,0.0,1.537438,What do you feel about that?,What do you feel about that?,1.537438,16000,1



DETECTED COLUMNS:
ID: uid
Audio: audio_path
Transcript: text
Split: split
Source: source
Duration: duration_s
Recording ID: recording_id

ROWS BY SOURCE:
source
openslr_mucs_slr104    55901
hiacc_adult             3318
hiacc_children          1858

ROWS BY SPLIT:
split
train         56391
test           4168
validation      518

DUPLICATE NON-EMPTY IDS: 0
MISSING IDS: 0

EMPTY TRANSCRIPTS: 0
WHITESPACE-ONLY / EMPTY TRANSCRIPTS: 0

DURATION AUDIT:
Unparseable/missing: 0
Zero or negative: 0
Total hours: 99.393
Min / median / max seconds: 0.37025 5.0 30.0

AUDIO PATH EXAMPLES:
['/content/drive/MyDrive/AI_PROJECTS/Code-SwitchingASR/data/Corpus/adult/audio/train_split/AD09002.wav', '/content/drive/MyDrive/AI_PROJECTS/Code-SwitchingASR/data/Corpus/adult/audio/train_split/AD09003.wav', '/content/drive/MyDrive/AI_PROJECTS/Code-SwitchingASR/data/Corpus/adult/audio/train_split/AD09005.wav', '/content/drive/MyDrive/AI_PROJECTS/Code-SwitchingASR/data/Corpus/adult/audio/train_split/AD09007.wav', '

In [21]:
import random
import json
import os
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import jiwer
from transformers import AutoProcessor, AutoModelForCTC

RUNS.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

REGISTRY_PATH = RESULTS / "experiment_registry.csv"

REGISTRY_COLUMNS = [
    "experiment_id",
    "phase",
    "model",
    "seed",
    "data_fraction",
    "encoder_strategy",
    "lid_weight",
    "config_json",
    "status",
    "start_time",
    "end_time",
    "metrics_json",
    "checkpoint_path",
]

if not REGISTRY_PATH.exists():
    pd.DataFrame(columns=REGISTRY_COLUMNS).to_csv(
        REGISTRY_PATH, index=False
    )

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def normalize_asr_text(text):
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    chars = []

    for char in text:
        if unicodedata.category(char).startswith("P"):
            chars.append(" ")
        else:
            chars.append(char)

    return " ".join("".join(chars).split())

def calculate_asr_metrics(references, predictions):
    if len(references) != len(predictions):
        raise ValueError("References and predictions must have equal lengths.")

    if len(references) == 0:
        raise ValueError("At least one reference-prediction pair is required.")

    raw_references = [str(x) for x in references]
    raw_predictions = [str(x) for x in predictions]

    normalized_references = [
        normalize_asr_text(x) for x in raw_references
    ]
    normalized_predictions = [
        normalize_asr_text(x) for x in raw_predictions
    ]

    return {
        "raw_wer": jiwer.wer(raw_references, raw_predictions),
        "normalized_wer": jiwer.wer(
            normalized_references, normalized_predictions
        ),
        "cer": jiwer.cer(
            normalized_references, normalized_predictions
        ),
    }

def save_checkpoint(
    path,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    epoch=0,
    step=0,
    best_metric=None,
    config=None,
):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": (
            optimizer.state_dict() if optimizer is not None else None
        ),
        "scheduler_state_dict": (
            scheduler.state_dict() if scheduler is not None else None
        ),
        "scaler_state_dict": (
            scaler.state_dict() if scaler is not None else None
        ),
        "epoch": epoch,
        "step": step,
        "best_metric": best_metric,
        "config": config or {},
        "python_rng_state": random.getstate(),
        "numpy_rng_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),
    }

    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(checkpoint, temporary_path)
    os.replace(temporary_path, path)

def load_checkpoint(
    path,
    model,
    optimizer=None,
    scheduler=None,
    scaler=None,
):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")

    checkpoint = torch.load(
        path, map_location="cpu", weights_only=False
    )

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None and checkpoint["optimizer_state_dict"] is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if scheduler is not None and checkpoint["scheduler_state_dict"] is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if scaler is not None and checkpoint["scaler_state_dict"] is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])

    random.setstate(checkpoint["python_rng_state"])
    np.random.set_state(checkpoint["numpy_rng_state"])
    torch.set_rng_state(checkpoint["torch_rng_state"])

    if torch.cuda.is_available() and checkpoint["cuda_rng_state"] is not None:
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state"])

    return {
        "epoch": checkpoint["epoch"],
        "step": checkpoint["step"],
        "best_metric": checkpoint["best_metric"],
        "config": checkpoint["config"],
    }

def register_experiment(
    experiment_id,
    phase,
    model_name,
    config,
    status="planned",
    metrics=None,
    checkpoint_path=None,
):
    registry = pd.read_csv(REGISTRY_PATH, dtype=str).fillna("")
    row = {
        "experiment_id": experiment_id,
        "phase": phase,
        "model": model_name,
        "seed": str(config.get("seed", 42)),
        "data_fraction": str(config.get("data_fraction", 1.0)),
        "encoder_strategy": str(config.get("encoder_strategy", "")),
        "lid_weight": str(config.get("lid_weight", "")),
        "config_json": json.dumps(config),
        "status": status,
        "start_time": "",
        "end_time": "",
        "metrics_json": json.dumps(metrics or {}),
        "checkpoint_path": str(checkpoint_path or ""),
    }

    registry = registry[
        registry["experiment_id"] != experiment_id
    ]
    registry = pd.concat(
        [registry, pd.DataFrame([row])],
        ignore_index=True,
    )
    registry.to_csv(REGISTRY_PATH, index=False)

    return registry

set_seed(42)

manifest = pd.read_csv(
    PROCESSED / "unified_manifest_hiacc_openslr.csv",
    low_memory=False,
)

assert {"train", "validation", "test"}.issubset(
    set(manifest["split"].astype(str).str.lower().unique())
), "Expected train, validation, and test splits."

test_metrics = calculate_asr_metrics(
    ["Hello, दुनिया!"],
    ["hello दुनिया"],
)

assert test_metrics["normalized_wer"] == 0.0
assert test_metrics["cer"] == 0.0

x = torch.randn(128, 128, device="cuda" if torch.cuda.is_available() else "cpu")
y = torch.randn(128, 128, device=x.device)

if torch.cuda.is_available():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        z = x @ y
else:
    z = x @ y

assert torch.isfinite(z).all().item()

print("Manifest rows:", len(manifest))
print("Experiment registry:", REGISTRY_PATH)
print("Checkpoint directory:", CHECKPOINTS)
print("Results directory:", RESULTS)
print("ASR metric smoke test:", test_metrics)
print("AutoProcessor available:", AutoProcessor is not None)
print("AutoModelForCTC available:", AutoModelForCTC is not None)
print("Mixed-precision smoke test: PASSED")

Manifest rows: 61077
Experiment registry: /content/drive/MyDrive/AI_PROJECTS/Code-SwitchingASR/data/experiments/results/experiment_registry.csv
Checkpoint directory: /content/drive/MyDrive/AI_PROJECTS/Code-SwitchingASR/data/experiments/checkpoints
Results directory: /content/drive/MyDrive/AI_PROJECTS/Code-SwitchingASR/data/experiments/results
ASR metric smoke test: {'raw_wer': 1.0, 'normalized_wer': 0.0, 'cer': 0.0}
AutoProcessor available: True
AutoModelForCTC available: True
Mixed-precision smoke test: PASSED
COMMON CELL 3 COMPLETE


## Phase A

### Setup and Configuration

In [1]:
import os
import sys
import gc
import io
import re
import random
import numpy as np
import torch
import soundfile as sf

from datasets import load_dataset
from scipy.signal import resample_poly
from math import gcd
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)



In [2]:
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
          "GB")
else:
    raise RuntimeError("Enable a GPU in Runtime > Change runtime type.")

from google.colab import userdata

try:
    HF_TOKEN = userdata.get("HF_API_KEY")
except Exception:
    HF_TOKEN = None

DATASET_NAME = "agarwalayushi/hinglish"

try:
    STREAM = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True,
        token=HF_TOKEN
    )
except Exception:
    STREAM = load_dataset(
        DATASET_NAME,
        split="train",
        streaming=True
    )

STREAM = STREAM.decode(False)

print("Dataset:", DATASET_NAME)
print("Columns:", STREAM.column_names)
print("Features:", STREAM.features)

Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA: True
GPU: Tesla T4
VRAM: 15.64 GB


Resolving data files:   0%|          | 0/816 [00:00<?, ?it/s]

Dataset: agarwalayushi/hinglish
Columns: ['source', 'text', 'audio', 'quality', 'duration']
Features: {'source': Value(dtype='string', id=None), 'text': Value(dtype='string', id=None), 'audio': Audio(sampling_rate=None, mono=True, decode=False, id=None), 'quality': Value(dtype='string', id=None), 'duration': Value(dtype='string', id=None)}
CELL 1 COMPLETE


### Collecting Data from HF Dataset: `agarwalayushi/hinglish`

In [3]:


TARGET_EXAMPLES = 3000
MAX_RECORDS_TO_SCAN = 60000
MIN_DURATION = 1.0
MAX_DURATION = 20.0

def clean_text(text):
    text = str(text or "")
    text = re.sub(r"</?hi-en>", " ", text, flags=re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def read_audio(record):
    audio = record.get("audio") or {}
    payload = audio.get("bytes")
    path = audio.get("path")

    if payload:
        samples, sr = sf.read(io.BytesIO(payload), dtype="float32")
    elif path and os.path.isfile(path):
        samples, sr = sf.read(path, dtype="float32")
    else:
        raise ValueError("Audio bytes/path unavailable")

    samples = np.asarray(samples, dtype=np.float32)

    if samples.ndim == 2:
        samples = samples.mean(axis=1)

    if samples.ndim != 1 or len(samples) == 0:
        raise ValueError("Invalid audio shape")

    if not np.isfinite(samples).all():
        raise ValueError("Non-finite audio values")

    if sr != 16000:
        g = gcd(int(sr), 16000)
        samples = resample_poly(
            samples, 16000 // g, int(sr) // g
        ).astype(np.float32)
        sr = 16000

    duration = len(samples) / sr

    if not MIN_DURATION <= duration <= MAX_DURATION:
        raise ValueError("Duration outside allowed range")

    return samples, sr, duration

# Shuffle across shards and within the stream.
shuffled_stream = STREAM.shuffle(
    seed=SEED,
    buffer_size=10000
)

RAW_EXAMPLES = []
errors = Counter()
sources = Counter()

for index, record in enumerate(shuffled_stream):
    if index >= MAX_RECORDS_TO_SCAN:
        break

    transcript = clean_text(record.get("text"))

    if not transcript:
        errors["empty_transcript"] += 1
        continue

    try:
        samples, sr, duration = read_audio(record)
    except Exception as e:
        errors[type(e).__name__ + ": " + str(e)[:70]] += 1
        continue

    source = str(record.get("source") or "unknown")

    RAW_EXAMPLES.append({
        "source": source,
        "text": transcript,
        "audio": samples,
        "sampling_rate": sr,
        "duration": duration,
    })
    sources[source] += 1

    if len(RAW_EXAMPLES) % 250 == 0:
        print(
            f"Scanned {index + 1:,} records | "
            f"valid clips: {len(RAW_EXAMPLES):,} | "
            f"distinct sources: {len(sources):,}"
        )

    if len(RAW_EXAMPLES) >= TARGET_EXAMPLES:
        break

print("\n========== DATA AUDIT ==========")
print("Records scanned:", min(index + 1, MAX_RECORDS_TO_SCAN))
print("Usable clips:", len(RAW_EXAMPLES))
print("Distinct sources:", len(sources))
print("Duration hours:",
      round(sum(x["duration"] for x in RAW_EXAMPLES) / 3600, 2))
print("Most common sources:", sources.most_common(10))
print("Decode/filter errors:", errors.most_common(10))

for i, item in enumerate(RAW_EXAMPLES[:5]):
    print(f"\nExample {i+1}")
    print("Source:", item["source"])
    print("Duration:", round(item["duration"], 2), "s")
    print("Transcript:", item["text"][:200])

if len(RAW_EXAMPLES) < 1000:
    raise RuntimeError(
        "Fewer than 1,000 usable clips were collected. "
        "Inspect the errors above before continuing."
    )

print("\nDATA COLLECTION PASSED")

Scanned 282 records | valid clips: 250 | distinct sources: 75
Scanned 552 records | valid clips: 500 | distinct sources: 92
Scanned 842 records | valid clips: 750 | distinct sources: 104
Scanned 1,137 records | valid clips: 1,000 | distinct sources: 111
Scanned 1,418 records | valid clips: 1,250 | distinct sources: 114
Scanned 1,691 records | valid clips: 1,500 | distinct sources: 119
Scanned 1,969 records | valid clips: 1,750 | distinct sources: 122


HTTP Error 503 thrown while requesting GET https://huggingface.co/datasets/agarwalayushi/hinglish/resolve/6aa2a2e6e07e63f84a5b2e388605372b06bbbb97/data/train-00259-of-00816.parquet
Retrying in 1s [Retry 1/5].


Scanned 2,258 records | valid clips: 2,000 | distinct sources: 125
Scanned 2,539 records | valid clips: 2,250 | distinct sources: 137
Scanned 2,822 records | valid clips: 2,500 | distinct sources: 140
Scanned 3,107 records | valid clips: 2,750 | distinct sources: 147
Scanned 3,400 records | valid clips: 3,000 | distinct sources: 152

========== DATA AUDIT ==========
Records scanned: 3400
Usable clips: 3000
Distinct sources: 152
Duration hours: 6.17
Most common sources: [('orpheus_tts_english_indian_male_malyalam_full_0000000001', 311), ('mkb_en_in__0000000001', 270), ('nptel_hin_JE_S5pZ-3JI', 68), ('nptel_hin_e8kyPe7napY', 68), ('nptel_hin_W10mueAYlAw', 60), ('nptel_hin_SBNzCUTq-fQ', 59), ('nptel_hin_i28BehHlH9M', 57), ('nptel_hin_VjAhv3gUatM', 56), ('nptel_hin_lFKQu3gee98', 49), ('nptel_hin_dBNLHXe2pO8', 48)]
Decode/filter errors: [('ValueError: Duration outside allowed range', 400)]

Example 1
Source: kathbath_hin_590
Duration: 4.87 s
Transcript: दोनों बुन्देलखण्ड इलाकों के बीच का अन

### Dataset Split on HF Dataset

In [4]:
from transformers import WhisperProcessor

MODEL_NAME = "openai/whisper-small"
LANGUAGE = "Hindi"

PROCESSOR = WhisperProcessor.from_pretrained(
    MODEL_NAME,
    language=LANGUAGE,
    task="transcribe"
)

# Prefer a source-level split to reduce leakage between train and evaluation.
unique_sources = list({x["source"] for x in RAW_EXAMPLES})
random.Random(SEED).shuffle(unique_sources)

if len(unique_sources) >= 10:
    n_eval_sources = max(1, round(0.10 * len(unique_sources)))
    eval_sources = set(unique_sources[:n_eval_sources])

    train_raw = [
        x for x in RAW_EXAMPLES if x["source"] not in eval_sources
    ]
    eval_raw = [
        x for x in RAW_EXAMPLES if x["source"] in eval_sources
    ]
else:
    # Fallback if source diversity is insufficient.
    shuffled = RAW_EXAMPLES.copy()
    random.Random(SEED).shuffle(shuffled)
    n_eval = max(100, round(0.10 * len(shuffled)))
    eval_raw = shuffled[:n_eval]
    train_raw = shuffled[n_eval:]
    print("WARNING: Few distinct sources; using a random clip-level split.")

print("Train clips:", len(train_raw))
print("Validation clips:", len(eval_raw))
print("Train sources:", len(set(x["source"] for x in train_raw)))
print("Validation sources:", len(set(x["source"] for x in eval_raw)))

def encode_record(item):
    features = PROCESSOR.feature_extractor(
        item["audio"],
        sampling_rate=16000,
        return_tensors="np"
    ).input_features[0]

    labels = PROCESSOR.tokenizer(item["text"]).input_ids

    # Remove the decoder-start token if present; the model adds it itself.
    if labels and labels[0] == PROCESSOR.tokenizer.bos_token_id:
        labels = labels[1:]

    return {
        "input_features": features.astype(np.float16),
        "labels": labels,
        "text": item["text"],
        "source": item["source"],
    }

def encode_many(items, name):
    output = []
    for i, item in enumerate(items):
        output.append(encode_record(item))
        if (i + 1) % 250 == 0:
            print(f"{name}: encoded {i+1}/{len(items)}")
    return output

TRAIN_ROWS = encode_many(train_raw, "Train")
EVAL_ROWS = encode_many(eval_raw, "Validation")

del RAW_EXAMPLES, train_raw, eval_raw
gc.collect()

from datasets import Dataset

train_dataset = Dataset.from_list(TRAIN_ROWS)
eval_dataset = Dataset.from_list(EVAL_ROWS)

del TRAIN_ROWS, EVAL_ROWS
gc.collect()

print("\nTrain dataset:", len(train_dataset))
print("Validation dataset:", len(eval_dataset))
print("Example transcript:", train_dataset[0]["text"])

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train clips: 2585
Validation clips: 415
Train sources: 137
Validation sources: 15
Train: encoded 250/2585
Train: encoded 500/2585
Train: encoded 750/2585
Train: encoded 1000/2585
Train: encoded 1250/2585
Train: encoded 1500/2585
Train: encoded 1750/2585
Train: encoded 2000/2585
Train: encoded 2250/2585
Train: encoded 2500/2585
Validation: encoded 250/415

Train dataset: 2585
Validation dataset: 415
Example transcript: दोनों बुन्देलखण्ड इलाकों के बीच का अन्तर खुली आंखों से देखा जा सकता है
CELL 3 COMPLETE


### Training

In [5]:

import torch
from dataclasses import dataclass
from typing import Any
from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)

assert len(train_dataset) >= 900
assert len(eval_dataset) >= 100

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)

model.config.use_cache = False
model.generation_config.language = "Hindi"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None

model.gradient_checkpointing_enable()

@dataclass
class WhisperCollator:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features):
        inputs = [
            {"input_features": np.asarray(x["input_features"], dtype=np.float32)}
            for x in features
        ]
        batch = self.processor.feature_extractor.pad(
            inputs, return_tensors="pt"
        )

        labels = [
            {"input_ids": x["labels"]} for x in features
        ]
        padded = self.processor.tokenizer.pad(
            labels, return_tensors="pt"
        )

        target_ids = padded["input_ids"].masked_fill(
            padded["attention_mask"].ne(1), -100
        )

        if (
            target_ids.shape[1] > 0
            and (target_ids[:, 0] == self.decoder_start_token_id).all().item()
        ):
            target_ids = target_ids[:, 1:]

        batch["labels"] = target_ids
        return batch

collator = WhisperCollator(
    processor=PROCESSOR,
    decoder_start_token_id=model.config.decoder_start_token_id
)

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return " ".join(text.split())

def compute_metrics(pred):
    from jiwer import wer

    pred_ids = pred.predictions
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    label_ids = pred.label_ids.copy()
    label_ids[label_ids == -100] = PROCESSOR.tokenizer.pad_token_id

    pred_text = PROCESSOR.tokenizer.batch_decode(
        pred_ids, skip_special_tokens=True
    )
    ref_text = PROCESSOR.tokenizer.batch_decode(
        label_ids, skip_special_tokens=True
    )

    pred_text = [normalize_text(x) for x in pred_text]
    ref_text = [normalize_text(x) for x in ref_text]

    return {"wer": wer(ref_text, pred_text)}

OUTPUT_DIR = "/content/whisper_hinglish_small"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_ratio=0.05,
    num_train_epochs=3,
    fp16=True,
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=1,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    save_total_limit=2,
    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
    compute_metrics=compute_metrics,
    processing_class=PROCESSOR,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting training...")
train_result = trainer.train()

print("\nTraining metrics:")
print(train_result.metrics)
print("\nBest checkpoint:", trainer.state.best_model_checkpoint)

Starting training...


Epoch,Training Loss,Validation Loss,Wer
0,0.499300,0.511366,0.736058
1,0.331400,0.469254,0.503342
2,0.235000,0.468123,0.489756


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instea


Training metrics:
{'train_runtime': 3042.854, 'train_samples_per_second': 2.549, 'train_steps_per_second': 0.159, 'total_flos': 2.23018797367296e+18, 'train_loss': 0.4024803302796484, 'epoch': 2.994586233565352}

Best checkpoint: /content/whisper_hinglish_small/checkpoint-483
CELL 4 COMPLETE


### Final Phase A Evaluation and Saving


In [6]:


from transformers import WhisperForConditionalGeneration
from jiwer import wer
import shutil

baseline_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME
).to("cuda")
baseline_model.eval()
baseline_model.config.use_cache = True

best_model = trainer.model
best_model.eval()
best_model.config.use_cache = True

def transcribe(model, features):
    inputs = torch.tensor(
        np.asarray(features, dtype=np.float32),
        dtype=torch.float32
    ).unsqueeze(0).to("cuda")

    with torch.inference_mode():
        output_ids = model.generate(
            input_features=inputs,
            max_length=128,
            num_beams=1,
            do_sample=False,
            language="Hindi",
            task="transcribe"
        )

    return PROCESSOR.batch_decode(
        output_ids, skip_special_tokens=True
    )[0].strip()

references = []
baseline_predictions = []
finetuned_predictions = []

for i, item in enumerate(eval_dataset):
    ref = item["text"]
    base = transcribe(baseline_model, item["input_features"])
    tuned = transcribe(best_model, item["input_features"])

    references.append(normalize_text(ref))
    baseline_predictions.append(normalize_text(base))
    finetuned_predictions.append(normalize_text(tuned))

    if i < 20:
        print(f"\n--- Example {i+1} ---")
        print("REFERENCE:", ref[:300])
        print("BASELINE: ", base[:300])
        print("FINE-TUNED:", tuned[:300])

baseline_wer = wer(references, baseline_predictions)
finetuned_wer = wer(references, finetuned_predictions)

print("\n========== FINAL RESULTS ==========")
print("Validation examples:", len(references))
print("Baseline WER:", round(baseline_wer, 4))
print("Fine-tuned WER:", round(finetuned_wer, 4))

if finetuned_wer < baseline_wer:
    print("RESULT: Fine-tuned model improved validation WER.")
else:
    print("RESULT: No validation WER improvement. Inspect predictions.")

FINAL_DIR = "/content/whisper_hinglish_final"
trainer.save_model(FINAL_DIR)
PROCESSOR.save_pretrained(FINAL_DIR)

print("\nSaved model and processor to:", FINAL_DIR)

archive_path = shutil.make_archive(
    "/content/whisper_hinglish_final",
    "zip",
    root_dir=FINAL_DIR
)

print("Model archive:", archive_path)

You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.



--- Example 1 ---
REFERENCE: After hockey good news for India also came in badminton
BASELINE:  अख़ाफ्त आप द़ाफ्ता और वो देखाफ्ता और वो देखाफ्ता और वो देखाफ्ता और वो देखाफ्ता और वो देखाफ्ता और वो देखाफ
FINE-TUNED: After hockey good news for India also came in the

--- Example 2 ---
REFERENCE: अपनी personal finance trackerओडीएस स्प्रैडशीट फाइल खोलते हैं
BASELINE:  अदिन परस्टनाल फाँनेज त्राकर ड़ोड एस श्प्रट शीट फाँल खोलते हैं
FINE-TUNED: अतनी परस्नल फ़ाइनेस ट्रैकर dot ods स्पेट शीट फाइल खोलते हैं

--- Example 3 ---
REFERENCE: इसमें कुछ ध्वनियाँ शामिल नहीं हैं जो अन्य भाषाओं में हैं
BASELINE:  अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अगर अग
FINE-TUNED: इसने कुछ दुहनिया शामिल लएई है जो अन्य भाषाओं में है

--- Example 4 ---
REFERENCE: If it is a video then share the video
BASELINE:  अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप अप
FINE-TUNED: इस इस विडियो दें शेय दे विडियो

--- Example 5 ---
REFE

## Phase B

### Installations

In [60]:
!pip -q install transformers==4.46.3 jiwer soundfile librosa scikit-learn accelerate


### Imports and Config

In [61]:
import os
import gc
import json
import re
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from google.colab import drive
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
)

drive.mount("/content/drive")

A_CHECKPOINT = Path(
    "/content/whisper_hinglish_small/checkpoint-483"
)

# If the checkpoint is stored in Google Drive instead,
# change A_CHECKPOINT to its actual Drive path.

assert A_CHECKPOINT.is_dir(), (
    f"Checkpoint not found: {A_CHECKPOINT}. "
    "Set A_CHECKPOINT to the actual saved checkpoint directory."
)

PROCESSOR = WhisperProcessor.from_pretrained(A_CHECKPOINT)

print("Checkpoint:", A_CHECKPOINT)
print("Architecture:", WhisperForConditionalGeneration.config_class
      .from_pretrained(A_CHECKPOINT).model_type)
print("Processor loaded.")
print("GPU available:", torch.cuda.is_available())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint: /content/whisper_hinglish_small/checkpoint-483
Architecture: whisper
Processor loaded.
GPU available: True


### Build Phase B model: saved Phase A Whisper + LID classification head


In [74]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import WhisperForConditionalGeneration

A_CHECKPOINT = "/content/whisper_hinglish_small/checkpoint-483"
LID_LABELS = ["hindi_script_dominant", "english_script_dominant", "mixed_script"]

class JointWhisperForConditionalGeneration(WhisperForConditionalGeneration):
    def __init__(self, config):
        super().__init__(config)
        self.lid_head = nn.Linear(config.d_model, len(LID_LABELS))

    def forward(self, input_features=None, labels=None, lid_labels=None, **kwargs):
        outputs = super().forward(
            input_features=input_features,
            labels=labels,
            output_hidden_states=False,
            return_dict=True,
            **kwargs
        )

        # Encoder representations: [batch, frames, hidden_size]
        encoder_outputs = self.model.encoder(
            input_features=input_features,
            return_dict=True
        )
        pooled = encoder_outputs.last_hidden_state.mean(dim=1)
        lid_logits = self.lid_head(pooled)

        lid_loss = None
        if lid_labels is not None:
            lid_loss = F.cross_entropy(
                lid_logits,
                lid_labels.long(),
                ignore_index=-100
            )

        # Return ASR and LID outputs together
        outputs["lid_logits"] = lid_logits
        outputs["lid_loss"] = lid_loss
        return outputs

joint_model = JointWhisperForConditionalGeneration.from_pretrained(A_CHECKPOINT)
joint_model.config.num_lid_labels = len(LID_LABELS)
joint_model.to("cuda" if torch.cuda.is_available() else "cpu")

# Freeze Whisper weights for a quick Phase B test; train only the LID head
for name, param in joint_model.named_parameters():
    param.requires_grad = name.startswith("lid_head.")

print("Model loaded:", joint_model.config.model_type)
print("Trainable parameters:", sum(p.numel() for p in joint_model.parameters() if p.requires_grad))
print("Device:", next(joint_model.parameters()).device)

Some weights of JointWhisperForConditionalGeneration were not initialized from the model checkpoint at /content/whisper_hinglish_small/checkpoint-483 and are newly initialized: ['lid_head.bias', 'lid_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: whisper
Trainable parameters: 2307
Device: cuda:0


### Generate full ASR + LID datasets from cached audio and transcript manifests


In [75]:


import re
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
from pathlib import Path
from torch.utils.data import Dataset
from transformers import WhisperProcessor

# Paths
EXP_DIR = Path(
    "/content/drive/MyDrive/AI_PROJECTS/Code-SwitchingASR/data/processed/experiments/v1"
)
A_CHECKPOINT = Path("/content/whisper_hinglish_small/checkpoint-483")

assert A_CHECKPOINT.exists(), f"Checkpoint not found: {A_CHECKPOINT}"

# Load the processor associated with Phase A
whisper_processor = WhisperProcessor.from_pretrained(str(A_CHECKPOINT))

# Load full train, validation, and test manifests
train_df = pd.read_csv(EXP_DIR / "train_10pct.csv")
val_df = pd.read_csv(EXP_DIR / "validation.csv")
test_df = pd.read_csv(EXP_DIR / "test.csv")

def get_lid_label(text):
    """0=Hindi-script dominant, 1=English-script dominant, 2=mixed script."""
    text = str(text)
    devanagari = len(re.findall(r"[\u0900-\u097F]", text))
    latin = len(re.findall(r"[A-Za-z]", text))
    total = devanagari + latin

    if total == 0:
        return -100

    latin_ratio = latin / total
    if latin_ratio <= 0.15:
        return 0
    elif latin_ratio >= 0.85:
        return 1
    return 2

def prepare_manifest(df, name):
    df = df.copy()

    # Prefer normalized transcripts when available
    if "text_normalized" in df.columns:
        normalized = df["text_normalized"].fillna("")
        df["text_norm"] = normalized.where(
            normalized.astype(str).str.strip().ne(""),
            df["text"].fillna("")
        ).astype(str).str.strip()
    else:
        df["text_norm"] = df["text"].fillna("").astype(str).str.strip()

    # Prefer cached, preprocessed audio
    if "cached_audio_path" in df.columns:
        cached = df["cached_audio_path"].fillna("").astype(str).str.strip()
        df["audio_path_use"] = cached.where(
            cached.ne(""), df["audio_path"].astype(str)
        )
    else:
        df["audio_path_use"] = df["audio_path"].astype(str)

    df["lid_label"] = df["text_norm"].apply(get_lid_label)

    # Remove unusable transcript/audio path rows
    df = df[
        df["text_norm"].str.len().gt(0)
        & df["audio_path_use"].str.len().gt(0)
        & df["lid_label"].ne(-100)
    ].reset_index(drop=True)

    print(f"{name}: {len(df):,} usable rows")
    print("  LID distribution:", df["lid_label"].value_counts().sort_index().to_dict())
    return df

train_df = prepare_manifest(train_df, "Train")
val_df = prepare_manifest(val_df, "Validation")
test_df = prepare_manifest(test_df, "Test")

class JointASRDataset(Dataset):
    def __init__(self, dataframe, processor, max_label_length=256):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor
        self.max_label_length = max_label_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = str(row["audio_path_use"])

        # Read cached audio
        audio, sr = sf.read(audio_path, dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        if sr != 16000:
            audio = librosa.resample(
                audio, orig_sr=sr, target_sr=16000
            ).astype(np.float32)

        # Whisper expects 16 kHz audio
        features = self.processor.feature_extractor(
            audio,
            sampling_rate=16000,
            return_tensors="np"
        ).input_features[0]

        # Tokenized ASR target
        token_ids = self.processor.tokenizer(
            str(row["text_norm"]),
            truncation=True,
            max_length=self.max_label_length
        ).input_ids

        return {
            "input_features": torch.tensor(features, dtype=torch.float32),
            "labels": torch.tensor(token_ids, dtype=torch.long),
            "lid_labels": torch.tensor(int(row["lid_label"]), dtype=torch.long),
        }

# Create full datasets; audio is loaded on demand rather than all into RAM
train_dataset = JointASRDataset(train_df, whisper_processor)
val_dataset = JointASRDataset(val_df, whisper_processor)
test_dataset = JointASRDataset(test_df, whisper_processor)

print("\nDatasets created successfully.")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Audio features and token labels are generated on demand.")


Train: 5,085 usable rows
  LID distribution: {0: 1219, 1: 269, 2: 3597}
Validation: 6,054 usable rows
  LID distribution: {0: 1335, 1: 292, 2: 4427}
Test: 4,168 usable rows
  LID distribution: {0: 1132, 1: 394, 2: 2642}

Datasets created successfully.
Train: 5085
Validation: 6054
Test: 4168
Audio features and token labels are generated on demand.


### Build a batch collator for joint Whisper ASR + LID training


In [76]:


from dataclasses import dataclass
from typing import Any
import numpy as np
import torch

@dataclass
class JointWhisperCollator:
    processor: Any

    def __call__(self, features):
        # Pad Whisper audio features to a batch
        audio_inputs = [
            {"input_features": f["input_features"].cpu().numpy()}
            for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            audio_inputs,
            return_tensors="pt"
        )

        # Pad transcript tokens; mask padding with -100 for ASR loss
        label_inputs = [
            {"input_ids": f["labels"].tolist()}
            for f in features
        ]
        padded_labels = self.processor.tokenizer.pad(
            label_inputs,
            return_tensors="pt"
        )

        labels = padded_labels["input_ids"].masked_fill(
            padded_labels["attention_mask"].ne(1), -100
        )

        # Remove decoder-start token if present in every label sequence
        decoder_start_id = self.processor.tokenizer.convert_tokens_to_ids(
            "<|startoftranscript|>"
        )
        if labels.shape[1] > 0 and (labels[:, 0] == decoder_start_id).all().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        batch["lid_labels"] = torch.stack(
            [f["lid_labels"] for f in features]
        )
        return batch

joint_collator = JointWhisperCollator(whisper_processor)



input_features (2, 80, 3000) torch.float32
labels (2, 31) torch.int64
lid_labels (2,) torch.int64


### Implement joint Whisper ASR + LID model with a combined loss


In [77]:


import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import WhisperForConditionalGeneration

LID_LOSS_WEIGHT = 0.3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class JointWhisperForConditionalGeneration(WhisperForConditionalGeneration):
    def __init__(self, config):
        super().__init__(config)
        self.lid_head = nn.Linear(config.d_model, 3)

    def forward(
        self,
        input_features=None,
        labels=None,
        lid_labels=None,
        **kwargs
    ):
        # Run Whisper's encoder once and reuse its representations for both tasks
        encoder_outputs = self.model.encoder(
            input_features=input_features,
            return_dict=True
        )

        asr_outputs = super().forward(
            input_features=input_features,
            encoder_outputs=encoder_outputs,
            labels=labels,
            return_dict=True,
            **kwargs
        )

        # Mean-pool encoder frames for utterance-level script classification
        pooled = encoder_outputs.last_hidden_state.mean(dim=1)
        lid_logits = self.lid_head(pooled)

        lid_loss = None
        total_loss = asr_outputs.loss

        if lid_labels is not None:
            lid_loss = F.cross_entropy(
                lid_logits,
                lid_labels.long(),
                ignore_index=-100
            )
            total_loss = asr_outputs.loss + LID_LOSS_WEIGHT * lid_loss

        return {
            "loss": total_loss,
            "asr_loss": asr_outputs.loss,
            "lid_loss": lid_loss,
            "logits": asr_outputs.logits,
            "lid_logits": lid_logits,
        }

# Reload the saved Phase A weights; the LID head starts fresh
joint_model = JointWhisperForConditionalGeneration.from_pretrained(A_CHECKPOINT)
joint_model.to(DEVICE)

# Fine-tune encoder + LID head; freeze decoder and output projection
for name, param in joint_model.named_parameters():
    param.requires_grad = (
        name.startswith("model.encoder.")
        or name.startswith("lid_head.")
    )

joint_model.config.use_cache = False

print("Model ready on:", DEVICE)
print("Trainable parameter count:",
      sum(p.numel() for p in joint_model.parameters() if p.requires_grad))
print("LID loss weight:", LID_LOSS_WEIGHT)


Some weights of JointWhisperForConditionalGeneration were not initialized from the model checkpoint at /content/whisper_hinglish_small/checkpoint-483 and are newly initialized: ['lid_head.bias', 'lid_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model ready on: cuda
Trainable parameter count: 88156419
LID loss weight: 0.3


### Full Phase B training: ASR + script-based LID


In [78]:


import os
import json
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

OUTPUT_DIR = "/content/drive/MyDrive/AI_PROJECTS/Code-SwitchingASR/outputs/phase_b_joint"
os.makedirs(OUTPUT_DIR, exist_ok=True)

EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8
ENCODER_LR = 1e-5
LID_LR = 1e-3
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=joint_collator,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=joint_collator,
    num_workers=2,
    pin_memory=True
)

optimizer = torch.optim.AdamW(
    [
        {
            "params": [
                p for n, p in joint_model.named_parameters()
                if p.requires_grad and not n.startswith("lid_head.")
            ],
            "lr": ENCODER_LR
        },
        {
            "params": [
                p for n, p in joint_model.named_parameters()
                if p.requires_grad and n.startswith("lid_head.")
            ],
            "lr": LID_LR
        }
    ],
    weight_decay=WEIGHT_DECAY
)

updates_per_epoch = int(np.ceil(len(train_loader) / GRAD_ACCUM))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS * updates_per_epoch
)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda"))

# Enable gradient checkpointing to reduce GPU memory use
joint_model.gradient_checkpointing_enable()
joint_model.config.use_cache = False

best_val_loss = float("inf")
history = []

for epoch in range(EPOCHS):
    joint_model.train()
    optimizer.zero_grad(set_to_none=True)

    running_total = 0.0
    running_asr = 0.0
    running_lid = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for step, batch in enumerate(progress):
        batch = {
            k: v.to(DEVICE, non_blocking=True)
            for k, v in batch.items()
        }

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            outputs = joint_model(**batch)
            loss = outputs["loss"] / GRAD_ACCUM

        scaler.scale(loss).backward()

        running_total += outputs["loss"].detach().float().item()
        running_asr += outputs["asr_loss"].detach().float().item()
        running_lid += outputs["lid_loss"].detach().float().item()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in joint_model.parameters() if p.requires_grad],
                MAX_GRAD_NORM
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        if (step + 1) % 100 == 0:
            progress.set_postfix(
                loss=f"{running_total/(step+1):.3f}",
                asr=f"{running_asr/(step+1):.3f}",
                lid=f"{running_lid/(step+1):.3f}"
            )

    train_metrics = {
        "train_loss": running_total / len(train_loader),
        "train_asr_loss": running_asr / len(train_loader),
        "train_lid_loss": running_lid / len(train_loader)
    }

    # Validation: combined loss and LID metrics; no test-set use
    joint_model.eval()
    val_total = val_asr = val_lid = 0.0
    val_lid_true, val_lid_pred = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            batch = {
                k: v.to(DEVICE, non_blocking=True)
                for k, v in batch.items()
            }

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):
                outputs = joint_model(**batch)

            val_total += outputs["loss"].float().item()
            val_asr += outputs["asr_loss"].float().item()
            val_lid += outputs["lid_loss"].float().item()

            preds = outputs["lid_logits"].argmax(dim=-1)
            valid = batch["lid_labels"] != -100
            val_lid_true.extend(batch["lid_labels"][valid].cpu().tolist())
            val_lid_pred.extend(preds[valid].cpu().tolist())

    val_metrics = {
        "val_loss": val_total / len(val_loader),
        "val_asr_loss": val_asr / len(val_loader),
        "val_lid_loss": val_lid / len(val_loader),
        "val_lid_accuracy": accuracy_score(val_lid_true, val_lid_pred),
        "val_lid_macro_f1": f1_score(
            val_lid_true, val_lid_pred, average="macro", zero_division=0
        )
    }

    epoch_record = {"epoch": epoch + 1, **train_metrics, **val_metrics}
    history.append(epoch_record)
    print(json.dumps(epoch_record, indent=2))

    # Save latest checkpoint and preserve best validation-loss checkpoint
    joint_model.save_pretrained(os.path.join(OUTPUT_DIR, "latest"))
    whisper_processor.save_pretrained(os.path.join(OUTPUT_DIR, "latest"))
    pd.DataFrame(history).to_csv(
        os.path.join(OUTPUT_DIR, "training_history.csv"), index=False
    )

    if val_metrics["val_loss"] < best_val_loss:
        best_val_loss = val_metrics["val_loss"]
        joint_model.save_pretrained(os.path.join(OUTPUT_DIR, "best"))
        whisper_processor.save_pretrained(os.path.join(OUTPUT_DIR, "best"))
        print(f"Saved best checkpoint at epoch {epoch+1}")

print("Training complete. Best validation loss:", best_val_loss)
print("Outputs:", OUTPUT_DIR)


Epoch 1/3:   0%|          | 0/2543 [00:00<?, ?it/s]

Validation:   0%|          | 0/3027 [00:00<?, ?it/s]

{
  "epoch": 1,
  "train_loss": 1.305477352098738,
  "train_asr_loss": 1.1164466641488808,
  "train_lid_loss": 0.6301022670984362,
  "val_loss": 0.9611953739132234,
  "val_asr_loss": 0.8214731699598478,
  "val_lid_loss": 0.4657406625788638,
  "val_lid_accuracy": 0.8201189296333002,
  "val_lid_macro_f1": 0.7350242268105335
}
Saved best checkpoint at epoch 1


Epoch 2/3:   0%|          | 0/2543 [00:00<?, ?it/s]

Validation:   0%|          | 0/3027 [00:00<?, ?it/s]

{
  "epoch": 2,
  "train_loss": 0.9347194368404241,
  "train_asr_loss": 0.7995887897026807,
  "train_lid_loss": 0.4504354730441821,
  "val_loss": 0.9184452838085648,
  "val_asr_loss": 0.7850324392938992,
  "val_lid_loss": 0.44470946340620776,
  "val_lid_accuracy": 0.8259002312520648,
  "val_lid_macro_f1": 0.7429057438914844
}
Saved best checkpoint at epoch 2


Epoch 3/3:   0%|          | 0/2543 [00:00<?, ?it/s]

Validation:   0%|          | 0/3027 [00:00<?, ?it/s]

{
  "epoch": 3,
  "train_loss": 0.8568188038296086,
  "train_asr_loss": 0.7554923278044117,
  "train_lid_loss": 0.3377549086693371,
  "val_loss": 0.9184940809911338,
  "val_asr_loss": 0.7819580488770875,
  "val_lid_loss": 0.455120090923177,
  "val_lid_accuracy": 0.8211100099108027,
  "val_lid_macro_f1": 0.7469470596570486
}
Training complete. Best validation loss: 0.9184452838085648
Outputs: /content/drive/MyDrive/AI_PROJECTS/Code-SwitchingASR/outputs/phase_b_joint


### Final Phase B evaluation and prediction export


In [83]:
class JointWhisperForConditionalGeneration(WhisperForConditionalGeneration):
    def __init__(self, config):
        super().__init__(config)
        self.lid_head = nn.Linear(config.d_model, 3)

    def forward(
        self,
        input_features=None,
        labels=None,
        lid_labels=None,
        encoder_outputs=None,
        **kwargs
    ):
        # During autoregressive generation, Whisper passes encoder_outputs
        # and decoder input_ids rather than input_features.
        if input_features is None:
            return super().forward(
                encoder_outputs=encoder_outputs,
                labels=labels,
                return_dict=True,
                **kwargs
            )

        # Normal joint training/evaluation path
        encoder_outputs = self.model.encoder(
            input_features=input_features,
            return_dict=True
        )

        asr_outputs = super().forward(
            input_features=None,
            encoder_outputs=encoder_outputs,
            labels=labels,
            return_dict=True,
            **kwargs
        )

        pooled = encoder_outputs.last_hidden_state.mean(dim=1)
        lid_logits = self.lid_head(pooled)

        lid_loss = None
        total_loss = asr_outputs.loss

        if lid_labels is not None:
            lid_loss = F.cross_entropy(
                lid_logits,
                lid_labels.long(),
                ignore_index=-100
            )
            total_loss = asr_outputs.loss + LID_LOSS_WEIGHT * lid_loss

        return {
            "loss": total_loss,
            "asr_loss": asr_outputs.loss,
            "lid_loss": lid_loss,
            "logits": asr_outputs.logits,
            "lid_logits": lid_logits,
        }

In [85]:


import os
import re
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from jiwer import wer, cer

BEST_DIR = os.path.join(OUTPUT_DIR, "best")
EVAL_DIR = os.path.join(OUTPUT_DIR, "evaluation")
os.makedirs(EVAL_DIR, exist_ok=True)

# Load Phase B joint model for LID
joint_model = JointWhisperForConditionalGeneration.from_pretrained(BEST_DIR)
joint_model.to(DEVICE)
joint_model.eval()

# Load standard Whisper from the same checkpoint for reliable ASR generation
asr_model = WhisperForConditionalGeneration.from_pretrained(BEST_DIR)
asr_model.to(DEVICE)
asr_model.eval()

whisper_processor = WhisperProcessor.from_pretrained(BEST_DIR)

LID_NAMES = [
    "hindi_script_dominant",
    "english_script_dominant",
    "mixed_script"
]

def normalize_eval_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return " ".join(text.split())

def evaluate_split(dataset, dataframe, split_name):
    loader = DataLoader(
        dataset,
        batch_size=2,
        shuffle=False,
        collate_fn=joint_collator,
        num_workers=2,
        pin_memory=(DEVICE.type == "cuda")
    )

    all_refs, all_hyps = [], []
    all_lid_true, all_lid_pred = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Evaluating {split_name}"):
            features = batch["input_features"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            lid_labels = batch["lid_labels"].to(DEVICE)

            # ASR generation uses the standard Whisper forward path
            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):
                generated_ids = asr_model.generate(
                    input_features=features,
                    max_new_tokens=128,
                    num_beams=1
                )

                # LID prediction uses the trained Phase B encoder and head
                encoder_outputs = joint_model.model.encoder(
                    input_features=features,
                    return_dict=True
                )
                pooled = encoder_outputs.last_hidden_state.mean(dim=1)
                lid_logits = joint_model.lid_head(pooled)

            labels_for_decode = labels.clone()
            labels_for_decode[labels_for_decode == -100] = (
                whisper_processor.tokenizer.pad_token_id
            )

            refs = whisper_processor.tokenizer.batch_decode(
                labels_for_decode, skip_special_tokens=True
            )
            hyps = whisper_processor.tokenizer.batch_decode(
                generated_ids, skip_special_tokens=True
            )

            all_refs.extend(normalize_eval_text(x) for x in refs)
            all_hyps.extend(normalize_eval_text(x) for x in hyps)

            lid_preds = lid_logits.argmax(dim=-1)
            valid = lid_labels != -100
            all_lid_true.extend(lid_labels[valid].cpu().tolist())
            all_lid_pred.extend(lid_preds[valid].cpu().tolist())

    if not all_refs:
        raise ValueError(f"No examples evaluated for {split_name}.")

    if len(all_lid_true) != len(all_refs):
        raise ValueError(
            f"Count mismatch: {len(all_refs)} ASR examples, "
            f"{len(all_lid_true)} valid LID labels."
        )

    metrics = {
        "split": split_name,
        "examples": len(all_refs),
        "WER": wer(all_refs, all_hyps),
        "CER": cer(all_refs, all_hyps),
        "LID_accuracy": accuracy_score(all_lid_true, all_lid_pred),
        "LID_macro_F1": f1_score(
            all_lid_true, all_lid_pred,
            average="macro", zero_division=0
        ),
        "LID_weighted_F1": f1_score(
            all_lid_true, all_lid_pred,
            average="weighted", zero_division=0
        )
    }

    predictions = pd.DataFrame({
        "reference": all_refs,
        "prediction": all_hyps,
        "lid_true": all_lid_true,
        "lid_pred": all_lid_pred
    })

    predictions.to_csv(
        os.path.join(EVAL_DIR, f"{split_name}_predictions.csv"),
        index=False
    )

    print(f"\n{split_name.upper()} metrics")
    print(pd.Series(metrics).to_string())

    print("\nLID classification report:")
    print(classification_report(
        all_lid_true,
        all_lid_pred,
        labels=[0, 1, 2],
        target_names=LID_NAMES,
        zero_division=0
    ))

    print("LID confusion matrix (true rows, predicted columns):")
    print(confusion_matrix(all_lid_true, all_lid_pred, labels=[0, 1, 2]))

    return metrics

# Evaluate validation and held-out test sets
val_results = evaluate_split(val_dataset, val_df, "validation")
test_results = evaluate_split(test_dataset, test_df, "test")

results_df = pd.DataFrame([val_results, test_results])
results_df.to_csv(
    os.path.join(EVAL_DIR, "final_metrics.csv"),
    index=False
)

print("\nSaved metrics and predictions to:", EVAL_DIR)
print(results_df.to_string(index=False))


Evaluating validation:   0%|          | 0/3027 [00:00<?, ?it/s]


VALIDATION metrics
split              validation
examples                 6054
WER                  0.589441
CER                  0.416353
LID_accuracy           0.8259
LID_macro_F1         0.742906
LID_weighted_F1      0.815495

LID classification report:
                         precision    recall  f1-score   support

  hindi_script_dominant       0.72      0.49      0.58      1335
english_script_dominant       0.71      0.80      0.75       292
           mixed_script       0.85      0.93      0.89      4427

               accuracy                           0.83      6054
              macro avg       0.76      0.74      0.74      6054
           weighted avg       0.82      0.83      0.82      6054

LID confusion matrix (true rows, predicted columns):
[[ 655   26  654]
 [   2  234   56]
 [ 248   68 4111]]


Evaluating test:   0%|          | 0/2084 [00:00<?, ?it/s]


TEST metrics
split                  test
examples               4168
WER                0.676345
CER                0.477349
LID_accuracy       0.768234
LID_macro_F1        0.74073
LID_weighted_F1     0.75718

LID classification report:
                         precision    recall  f1-score   support

  hindi_script_dominant       0.68      0.48      0.56      1132
english_script_dominant       0.82      0.84      0.83       394
           mixed_script       0.78      0.88      0.83      2642

               accuracy                           0.77      4168
              macro avg       0.76      0.73      0.74      4168
           weighted avg       0.76      0.77      0.76      4168

LID confusion matrix (true rows, predicted columns):
[[ 539   11  582]
 [   2  332   60]
 [ 249   62 2331]]

Saved metrics and predictions to: /content/drive/MyDrive/AI_PROJECTS/Code-SwitchingASR/outputs/phase_b_joint/evaluation
     split  examples      WER      CER  LID_accuracy  LID_macro_F1  LID_wei